In [3]:
# set the project root and load the finished SROIE modeling table
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

model_df_path = PROJECT_ROOT / "outputs" / "sroie_model_df.csv"
sroie_model_df = pd.read_csv(model_df_path)

print(model_df_path)
print(sroie_model_df.shape)
display(sroie_model_df.head())

/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/outputs/sroie_model_df.csv
(712, 46)


,doc_id,dataset,split,img_width,img_height,n_tokens,n_boxes,ocr_char_count,ocr_word_count,company_present,...,ocr_is_empty,aspect_ratio,company_hard,date_hard,address_hard,total_hard,low_ocr_support,proxy_risk_score,proxy_verify,proxy_high_risk
0,X00016469612,SROIE,train,463.0,1013.0,44,44,441,84,True,...,False,0.457058,True,False,False,False,False,1,False,False
1,X00016469619,SROIE,train,439.0,1004.0,48,48,637,102,True,...,False,0.437251,False,False,False,False,False,0,False,False
2,X00016469620,SROIE,train,459.0,949.0,54,54,668,121,True,...,False,0.483667,True,False,False,True,False,2,True,False
3,X00016469622,SROIE,train,461.0,933.0,60,60,525,105,True,...,False,0.494105,False,False,False,False,False,0,False,False
4,X00016469623,SROIE,train,463.0,1026.0,61,61,735,142,True,...,False,0.451267,False,False,False,True,False,1,False,False


In [8]:
# inspect the proxy-label balance that all baselines will be compared against
display(
    sroie_model_df[
        [
            "proxy_risk_score",
            "proxy_verify",
            "proxy_high_risk",
            "company_hard",
            "date_hard",
            "address_hard",
            "total_hard",
            "low_ocr_support",
        ]
    ]
    .mean(numeric_only=True)
    .to_frame("mean_or_rate")
)

display(sroie_model_df["proxy_risk_score"].value_counts().sort_index())

,mean_or_rate
proxy_risk_score,0.398876
proxy_verify,0.074438
proxy_high_risk,0.063202
company_hard,0.080056
date_hard,0.067416
address_hard,0.116573
total_hard,0.122191
low_ocr_support,0.012640


proxy_risk_score
0    575
1     84
2      8
3      2
4     37
5      6
Name: count, dtype: int64

In [9]:
# define simple baseline verification policies from the feature table
baseline_df = sroie_model_df[["doc_id", "proxy_verify"]].copy()

baseline_df["always_accept"] = False
baseline_df["always_verify"] = True

baseline_df["verify_if_any_missing"] = sroie_model_df["any_field_missing"]
baseline_df["verify_if_ocr_empty"] = sroie_model_df["ocr_is_empty"]
baseline_df["verify_if_total_not_in_ocr"] = ~sroie_model_df["total_in_ocr"]
baseline_df["verify_if_address_not_in_ocr"] = ~sroie_model_df["address_in_ocr"]
baseline_df["verify_if_company_not_in_ocr"] = ~sroie_model_df["company_in_ocr"]
baseline_df["verify_if_date_not_in_ocr"] = ~sroie_model_df["date_in_ocr"]
baseline_df["verify_if_total_ambiguous"] = sroie_model_df["exact_total_matches"] == 0
baseline_df["verify_if_low_ocr_support"] = sroie_model_df["low_ocr_support"]
baseline_df["verify_if_amount_heavy"] = sroie_model_df["n_amount_like_tokens"] >= 20
baseline_df["verify_if_two_or_more_hard_flags"] = (
    sroie_model_df[["company_hard", "date_hard", "address_hard", "total_hard"]]
    .sum(axis=1) >= 2
)

display(baseline_df.head())

,doc_id,proxy_verify,always_accept,always_verify,verify_if_any_missing,verify_if_ocr_empty,verify_if_total_not_in_ocr,verify_if_address_not_in_ocr,verify_if_company_not_in_ocr,verify_if_date_not_in_ocr,verify_if_total_ambiguous,verify_if_low_ocr_support,verify_if_amount_heavy,verify_if_two_or_more_hard_flags
0,X00016469612,False,False,True,False,False,False,False,True,False,False,False,False,False
1,X00016469619,False,False,True,False,False,False,False,False,False,False,False,False,False
2,X00016469620,True,False,True,False,False,False,False,True,False,True,False,False,True
3,X00016469622,False,False,True,False,False,False,False,False,False,False,False,False,False
4,X00016469623,False,False,True,False,False,False,False,False,False,True,False,False,False


In [10]:
# evaluate each baseline against the current proxy-verify label
def baseline_metrics(y_true: pd.Series, y_pred: pd.Series) -> dict:
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)

    tp = int((y_true & y_pred).sum())
    tn = int((~y_true & ~y_pred).sum())
    fp = int((~y_true & y_pred).sum())
    fn = int((y_true & ~y_pred).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    accuracy = (tp + tn) / len(y_true) if len(y_true) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    pred_rate = y_pred.mean()

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "f1": f1,
        "pred_verify_rate": pred_rate,
    }

results = []
target = baseline_df["proxy_verify"]

for col in baseline_df.columns:
    if col in ["doc_id", "proxy_verify"]:
        continue
    metrics = baseline_metrics(target, baseline_df[col])
    metrics["baseline"] = col
    results.append(metrics)

baseline_results_df = pd.DataFrame(results).sort_values(
    ["f1", "recall", "precision"], ascending=False
)

display(baseline_results_df)

,tp,tn,fp,fn,precision,recall,specificity,accuracy,f1,pred_verify_rate,baseline
11,53,659,0,0,1.000000,1.000000,1.000000,1.000000,1.000000,0.074438,verify_if_two_or_more_hard_flags
4,45,658,1,8,0.978261,0.849057,0.998483,0.987360,0.909091,0.064607,verify_if_total_not_in_ocr
7,45,656,3,8,0.937500,0.849057,0.995448,0.984551,0.891089,0.067416,verify_if_date_not_in_ocr
2,43,657,2,10,0.955556,0.811321,0.996965,0.983146,0.877551,0.063202,verify_if_any_missing
6,48,648,11,5,0.813559,0.905660,0.983308,0.977528,0.857143,0.082865,verify_if_company_not_in_ocr
5,52,626,33,1,0.611765,0.981132,0.949924,0.952247,0.753623,0.119382,verify_if_address_not_in_ocr
8,51,623,36,2,0.586207,0.962264,0.945372,0.946629,0.728571,0.122191,verify_if_total_ambiguous
3,8,659,0,45,1.000000,0.150943,1.000000,0.936798,0.262295,0.011236,verify_if_ocr_empty
9,8,658,1,45,0.888889,0.150943,0.998483,0.935393,0.258065,0.012640,verify_if_low_ocr_support
1,53,0,659,0,0.074438,1.000000,0.000000,0.074438,0.138562,1.000000,always_verify
